# Multi-agent TD3 for MPE (Multi-Agent Particle Environment)

* MATD3: Reducing Overestimation Bias in Multi-Agent Domains using Double Centralized Critics (MATD3 2019)[paper](https://arxiv.org/abs/1910.01465)
* MPE: Multi-Agent Particle Environment using pettingzoo library [github](https://github.com/Farama-Foundation/PettingZoo)
* TD3: Twin Delayed Deep Deterministic Policy Gradient (2018) [paper](https://arxiv.org/abs/1802.09477)

## MATD3 Algorithm

**MATD3** extends Twin Delayed Deep Deterministic Policy Gradient (TD3) to multi-agent environments, combining TD3's variance reduction techniques with MADDPG's centralized training approach.

### Key Components

MATD3 incorporates the following enhancements over MADDPG:

* **Twin Centralized Critics**: Each agent maintains two centralized Q-functions to reduce overestimation bias
* **Delayed Policy Updates**: Actor networks update less frequently than critics (typically every 2 steps)
* **Target Policy Smoothing**: Add clipped noise to target actions for regularization
* **Clipped Double Q-learning**: Use the minimum of the twin critics when computing target values

### Twin Critics Update

For each agent i, maintain two critics $Q^{\vec{\mu}}_{i,1}$ and $Q^{\vec{\mu}}_{i,2}$:

$$ 
\begin{aligned} 
\mathcal{L}(\phi_{i,k}) &= \mathbb{E}_{\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}'}[ (Q^{\vec{\mu}}_{i,k}(\vec{o}, a_1, \dots, a_N) - y_i)^2 ] & \text{for } k \in {1,2} & \\

\text{where } y_i &= r_i + \gamma \min_{k=1,2} Q^{\vec{\mu}'}_{i,k} (\vec{o}', a'_1, \dots, a'N) \rvert{a'_j = \mu'_{\theta_j}(o'_j) + \epsilon} & \  \text{for } \epsilon &\sim \text{clip}(\mathcal{N}(0, \sigma), -c, c) \end{aligned} 
$$

where $\vec{\mu}’$ are the target policies with delayed softly-updated parameters.

### Delayed Actor Update

Update each agent's policy less frequently (every d steps):

$$ 
\nabla_{\theta_i} J(\theta_i) = \mathbb{E}{\vec{o}, \vec{a} \sim \mathcal{D}} [\nabla{a_i} Q^{\vec{\mu}}_{i,1} (\vec{o}, a_1, \dots, a_N) \nabla{\theta_i} \mu_{\theta_i}(o_i) \rvert_{a_i=\mu_{\theta_i}(o_i)} ] \
$$

Where $\mathcal{D}$ is the memory buffer for experience replay, containing multiple episode samples $(\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}’)$ — given current observation $\vec{o}$, agents take action $a_1, \dots, a_N$ and get rewards $r_1, \dots, r_N$, leading to the new observation $\vec{o}’$. 

Sometimes I prefer to take min of the two critics to reduce overestimation bias in the actor update step. It is not mentioned in the paper but it is a common practice in TD3.

$$
\min_{k=1,2} Q^{\vec{\mu}}_{i,k} (\vec{o}, a_1, \dots, a_N)
$$

### Advantages over MADDPG

1. **Reduced Overestimation**: Twin critics combat the overestimation bias that's amplified in multi-agent settings
2. **More Stable Learning**: Delayed policy updates prevent premature policy convergence
3. **Improved Robustness**: Target policy smoothing makes value estimates less sensitive to specific actions
4. **Better Cooperation**: More accurate Q-value estimates lead to better coordinated policies

Here is the final algorithm:


<div style="text-align:center"><img src="../../assets/images/MATD3-algorithm.png" width="600" height="auto"></div>
